In [48]:
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os
import re
from pathlib import Path
import subprocess
import tempfile



load_dotenv()

True

In [49]:
OUT_MERGED_CSV = f"{os.getenv('PROJECT_ROOT_DIR')}/dataset"

In [50]:
df_list_path = f"{OUT_MERGED_CSV}/manual-malicious.csv"

In [51]:
df_list = pd.read_csv(df_list_path)

In [52]:
df_list = df_list[df_list["label"]=="Malicious"]

In [ ]:
def _remote_dir_exists(remote_host: str, remote_dir: str) -> bool:
    cmd = [
        "ssh",
        remote_host,
        f'test -d "{remote_dir}"'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0



In [ ]:
def rsync_hash_dirs_with_check(
    df_list,
    remote_host: str,
    column: str = "identifier",
    remote_base_dir: str = "/syssec-nas0/pyc-research/decompiler_workspace",
    local_base_dir: str | None = None,
    dry_run: bool = False,
    verbose: bool = True,
    fail_if_missing: bool = False,
):
    if local_base_dir is None:
        local_base_dir = f"{os.getenv('ROOT_FOR_FILES')}/malware_files"

    identifiers = (
        df_list[column]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    if not identifiers:
        raise ValueError(f"No valid identifiers found in df_list['{column}'].")

    Path(local_base_dir).mkdir(parents=True, exist_ok=True)

    existing = []
    missing = []

    for h in identifiers:
        remote_dir = f"{remote_base_dir.rstrip('/')}/{h}"
        if _remote_dir_exists(remote_host, remote_dir):
            existing.append(h)
        else:
            missing.append(h)

    if verbose:
        print(f"Found {len(existing)} existing remote folders.")
        print(f"Missing {len(missing)} remote folders.")
        if missing:
            print("\nMissing examples:")
            for h in missing[:10]:
                print(f"  {remote_base_dir.rstrip('/')}/{h}")

    if fail_if_missing and missing:
        raise FileNotFoundError(
            f"{len(missing)} remote folders do not exist. "
            f"Examples: {missing[:5]}"
        )

    if not existing:
        if verbose:
            print("No existing remote folders to download.")
        return {
            "downloaded": [],
            "missing": missing,
            "rsync_stdout": "",
            "rsync_stderr": "",
        }

    with tempfile.NamedTemporaryFile(mode="w", delete=False, encoding="utf-8") as tf:
        for h in existing:
            tf.write(h + "\n")
        files_from_path = tf.name

    cmd = [
        "rsync",
        "-avz",
        "--files-from",
        files_from_path,
    ]

    if dry_run:
        cmd.append("--dry-run")

    cmd.extend([
        f"{remote_host}:{remote_base_dir.rstrip('/')}/",
        str(Path(local_base_dir)),
    ])

    if verbose:
        print("\nRunning rsync:")
        print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    if verbose:
        print("\nSTDOUT:\n", result.stdout)
        print("\nSTDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"rsync failed with code {result.returncode}")

    return {
        "downloaded": existing,
        "missing": missing,
        "rsync_stdout": result.stdout,
        "rsync_stderr": result.stderr,
    }

In [55]:
DOWNLOAD_PATH = f"{os.getenv('ROOT_FOR_FILES')}/malware_files/"
REMOTE_PATH="/syssec-nas1/pyc-research/decompiler_workspace"
REMOTE_HOST="mxs220189@syssec-pyuser2"

In [ ]:
result = rsync_hash_dirs_with_check(
    df_list=df_list,
    remote_host=REMOTE_HOST,
    local_base_dir=DOWNLOAD_PATH,
    remote_base_dir=REMOTE_PATH,
    dry_run=False,
    verbose=True
)

Found 610 existing remote folders.
Missing 1408 remote folders.

Missing examples:
  /syssec-nas0/pyc-research/decompiler_workspace/012328b41355df983abc48bba1e1ddbec55bd670130572a479694107fa90fff4
  /syssec-nas0/pyc-research/decompiler_workspace/294bac9f0b4b8756077a3883537b89287ba5694aa864650067644cdba9cb8a81
  /syssec-nas0/pyc-research/decompiler_workspace/4a212583754332f7a88661b11802d0f0d8bdcbaea0dffb4ad8b828e5bf8ee653
  /syssec-nas0/pyc-research/decompiler_workspace/51b6c2dcd1220489b6e185034995de198accb8c9865714ec0792b93c370a8cf5
  /syssec-nas0/pyc-research/decompiler_workspace/542732b5c4ec6cbda8bfcaeac05d8f2ec22f9330cbe61a88425e2b4b0a721e97
  /syssec-nas0/pyc-research/decompiler_workspace/5d11be724b4cfc3c3d95f56c851fcdba38f73ca977c71b407b46b726c2adc36a
  /syssec-nas0/pyc-research/decompiler_workspace/80097d00c72151ba5c46e935e37b72db24a0130c71307c2cec5fc0515873d644
  /syssec-nas0/pyc-research/decompiler_workspace/99c2287758b6a197648211995e3694c54b9bb6432447b73a84e5207148e04f2c
  /sy

In [57]:
print("Downloaded:", len(result["downloaded"]))
print("Missing:", len(result["missing"]))

Downloaded: 610
Missing: 1408
